# 02 — Preprocessing, splitting and leakage prevention

**Purpose.** Produce leakage-free train/validation/test splits and fit the preprocessing pipeline on the training
split only (ML-01, ML-02).

| | |
|---|---|
| **Inputs** | raw CSVs (same loader as notebook 01) |
| **Outputs** | `data/processed/<family>/{train,validation,test}.parquet`, `preprocessor.joblib`, `preprocessing_report.json` |

Order of operations: deduplicate → split → remove cross-split duplicates → choose features on **train** → fit the
preprocessor on **train** → save. The fitted preprocessor is later packaged into the model bundle, and the API applies
it unchanged at inference time.

In [ ]:
# --- Project bootstrap: make backend/app/ml and notebooks/nb_utils importable ---------------------
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not ((_root / "backend" / "app" / "ml").is_dir() and (_root / "notebooks").is_dir()):
    if _root.parent == _root:
        raise RuntimeError("Start Jupyter inside the CyberSentinel repository (for example from notebooks/).")
    _root = _root.parent
for _path in (str(_root / "backend"), str(_root / "notebooks")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

from nb_utils import load_config

CFG = load_config()  # all settings come from CS_* environment variables (see nb_utils/config.py)
CFG.describe()

In [ ]:
import warnings

import numpy as np
import pandas as pd

from app.ml.constants import BENIGN_LABEL
from app.ml.datasets import get_schema_for_family
from nb_utils import io, raw_data, splits
from nb_utils.experiments import save_json
from nb_utils.preprocessing import PREPROCESSING_STEPS, benign_reference, build_preprocessor, input_feature_specs

warnings.filterwarnings("ignore", category=FutureWarning)
SCHEMA = get_schema_for_family(CFG.family)
PREPROCESSING_VERSION = f"{SCHEMA.family}-prep-1.0.0"
LABEL = "label_category"

In [ ]:
data, load_report = raw_data.load_family(CFG.raw_dir, CFG.family, max_rows_per_file=CFG.max_rows_per_file)
FEATURES_ALL = SCHEMA.feature_names
print(data.shape)

## 1. Deduplicate (before splitting)

In [ ]:
strategy = (
    "official UNSW-NB15 testing partition as test set; validation = 15% stratified sample of the official training partition"
    if CFG.family == "unsw_nb15"
    else "stratified random split by category (70/15/15) after deduplication"
)
split_report = splits.SplitReport(strategy=strategy, seed=CFG.seed)
data = splits.deduplicate(data, FEATURES_ALL, LABEL, split_report)
print("exact duplicates removed:", split_report.exact_duplicates_removed)
print("rows removed because identical features carried conflicting labels:", split_report.conflicting_rows_removed)
print("remaining rows:", len(data))

## 2. Split
* **CIC-IDS2017** has no official split; a time/day-based split is not usable because several attack types occur on a
  single day only, so a stratified random split is used after deduplication.
* **UNSW-NB15** keeps the publisher's testing partition untouched; validation is carved out of the training partition.

Classes too rare to stratify stay in the training split and are reported (their test metrics will be undefined).

In [ ]:
if CFG.family == "unsw_nb15":
    train, val, test = splits.official_test_split(data, LABEL, val_size=0.15, seed=CFG.seed, report=split_report)
else:
    train, val, test = splits.stratified_three_way(data, LABEL, val_size=0.15, test_size=0.15, seed=CFG.seed, report=split_report)
print("rare classes kept in training only:", split_report.rare_classes_train_only)
print({"train": len(train), "validation": len(val), "test": len(test)})

## 3. Remove cross-split duplicates and assert disjointness

In [ ]:
val = splits.remove_cross_split_duplicates(train, val, FEATURES_ALL, "validation_vs_train", split_report)
test = splits.remove_cross_split_duplicates(pd.concat([train, val]), test, FEATURES_ALL, "test_vs_train_and_validation", split_report)
splits.assert_disjoint(train, val, test, FEATURES_ALL)
print("cross-split duplicates removed:", split_report.cross_split_duplicates_removed)

## 4. Feature selection (training split only)

In [ ]:
dropped = {c: "constant in the training split" for c in FEATURES_ALL if train[c].nunique(dropna=True) <= 1}
if CFG.drop_shortcut_features:
    dropped.update({c: "shortcut-feature ablation (CS_DROP_SHORTCUT_FEATURES=1)" for c in SCHEMA.shortcut_feature_candidates})
FEATURES = [c for c in FEATURES_ALL if c not in dropped]
print(f"{len(FEATURES)} input features kept; dropped: {dropped}")

## 5. Fit the preprocessing pipeline on the training split only

In [ ]:
specs = input_feature_specs(SCHEMA, FEATURES, train)
preprocessor = build_preprocessor(specs)
preprocessor.fit(train[FEATURES])
transformed_names = list(preprocessor.get_feature_names_out())
print("input features:", preprocessor.n_features_in_, "| transformed features:", len(transformed_names))
assert np.isfinite(preprocessor.transform(val[FEATURES].head(2000))).all(), "non-finite values after preprocessing"
scaler = preprocessor.named_transformers_["num"].named_steps["scale"] if "num" in preprocessor.named_transformers_ else None

## 6. Benign reference statistics
Median/IQR (numeric) and mode (categorical) of **benign training** rows; the API uses them to phrase reason codes ("value is unusually high compared with benign traffic").

In [ ]:
reference = benign_reference(train, specs, train[LABEL] == BENIGN_LABEL)
print(len(reference), "features have benign reference statistics")

## 7. Save splits, preprocessor and report

In [ ]:
keep = FEATURES_ALL + ["label_original", "label_fine", "label_category", "label_binary", "source_file", "source_row", "official_split"]
io.save_splits({"train": train[keep], "validation": val[keep], "test": test[keep]}, CFG.processed_dir)
io.save_object(preprocessor, CFG.processed_dir / "preprocessor.joblib")
splits.record_class_counts(split_report, {"train": train, "validation": val, "test": test}, LABEL)
report = {
    "family": CFG.family,
    "schema_version": SCHEMA.schema_version,
    "preprocessing_version": PREPROCESSING_VERSION,
    "preprocessing_steps": PREPROCESSING_STEPS,
    "input_features": [s.model_dump() for s in specs],
    "dropped_features": dropped,
    "transformed_feature_count": len(transformed_names),
    "benign_reference": {k: v.model_dump() for k, v in reference.items()},
    "split": split_report.to_dict(),
    "raw_files": load_report.files,
    "label_mapping_version": SCHEMA.label_mapping_version,
    "seed": CFG.seed,
}
save_json(report, CFG.processed_dir / "preprocessing_report.json")
save_json(report, CFG.reports_dir / "preprocessing_report.json")
pd.DataFrame(split_report.class_counts).fillna(0).astype(int)

## 8. Leakage checklist (ML-01)

In [ ]:
checks = {
    "deduplicated before splitting": split_report.rows_before_dedup >= len(train) + len(val) + len(test),
    "train/validation/test share no feature vectors": True,  # asserted in section 3
    "label columns are not features": not set(FEATURES) & {"label_original", "label_fine", "label_category", "label_binary"},
    "identifier columns are not features": not set(FEATURES) & set(SCHEMA.identifier_columns),
    "scaler fitted on the training split only": scaler is None or int(scaler.n_samples_seen_) == len(train),
}
assert all(checks.values()), checks
pd.Series(checks, name="passed")